<img src="logo.png" alt="Vegeta" width="240">

# MERLIN — rapid atmospheric sampling over a wildfire (26)

MERLIN is a compact electric fixed-wing that leaves a launch stand, **dashes** to a reported hotspot, flies crosswind
passes through the smoke column with a forward gas sensor (CO, CO₂, particles), and comes home for a belly landing.
Two jobs: **rapid sampling** of a plume, a smoke column or a gas release, and **ignition confirmation** — is the
reported hotspot a real fire, and how big is it.

The question this notebook answers: **which propulsor gets MERLIN to a fire 5, 8, 10, 20 or 30 km from the launch
site fastest** — a nose **ducted fan** (the jet leaves as a ring around the fuselage), a **tractor** propeller on the nose
or a **pusher** propeller behind the tail — on the same battery and the same power limit, and still home with a reserve.

Nothing is designed twice. The airframe is notebook 09's `FixedWing` (its wing, tail and sections; `designs/merlin.py`
subclasses it), the ducted fan is notebook 25's EDF (`designs/ducted_fan.py`, `boreas.ducted`, its losses fitted to the
same catalogue point), the propellers are notebook 25's blade with its installation models (`designs/air_propeller.py`:
the effective wake and the thrust deduction), and the mission, the smoke and the movie are in `designs/merlin_flight.py`.

```
Part 1  airframe:  mission + mass budget → the three MERLINs in CAD → drag build-up → wing FEA (pull-up, dash gust), two revisions
                   → whole-aircraft RANS per variant (optional) → print the nose unit
Part 2  the race:  the three propulsors (imported models) → thrust and power against speed → time to the fire at 5 / 8 / 10 / 20 / 30 km
                   → the winner, what limits it, where the sensor breathes
Part 3  in flight: the winner with its propeller as a rotor disk (CFD, optional) → the mission through a smoke plume
                   → the fire's size from the passes → the movie with the smoke → hand-off
```

The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them; the drag build-up is the polar then).
`MERLIN_QUICK=1` uses coarse propulsor tables and a one-second movie (a check that everything runs, not results).

In [ ]:
import json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video, display
from vegeta import dedalus, talos, aeromant, mellonia, core, boreas
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
sys.path.insert(0, "designs")
import merlin as mdesign
import merlin_flight as mf

ROOT = Path("_runs/merlin"); shutil.rmtree(ROOT, ignore_errors=True); ROOT.mkdir(parents=True)
OUT = Path("output") / "26_merlin"; OUT.mkdir(parents=True, exist_ok=True)       # every movie of this notebook
RHO, NU, G = 1.225, 1.5e-5, 9.81
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUICK = os.environ.get("MERLIN_QUICK") == "1"
KINDS = ("edf", "tractor", "pusher")
NAME = {"edf": "ducted fan", "tractor": "tractor propeller", "pusher": "pusher propeller"}
COLOR = {"edf": "#9467bd", "tractor": "#1f77b4", "pusher": "#d62728"}
print(f"CFD {'on' if RUN_CFD else 'skipped'}; {'QUICK check' if QUICK else 'full tables'}")

# Part 1 — the airframe

## 1. Mission and mass budget — edit here

The three MERLINs share everything except the propulsor: the same printed airframe, the same 6S 8000 mAh battery, the
same flight controller and the same gas sensor. Each propulsor brings its own mass: the EDF unit (its motor inside,
320 g) with an 80 A controller, or a 10-inch propeller with an outrunner and a 60 A controller. **One electrical power
limit for all**: 1900 W, the EDF's catalogue full power (about 12 C from this pack) — so the race compares propulsors,
not batteries or motors.

In [ ]:
MISSION = mf.Mission(sampling_agl_m=150.0, sampling_s=180.0, launch_speed=12.0, battery_wh=6 * 3.7 * 8.0,
                     usable_fraction=0.85, reserve_fraction=0.15)
DISTANCES_KM = mf.DISTANCES_KM                        # 5, 8, 10, 20, 30 km
MAX_ELECTRICAL_W, DRIVE_EFF = 1900.0, 0.85
COMMON = dict(mf.COMMON_MASS_KG)                      # kg; the airframe line is replaced from the CAD below
UNIT_MASS = dict(mf.UNIT_MASS_KG)
SHELL_AREAL_DENSITY_G_MM2 = 0.7e-3                    # 0.7 kg/m^2: ~1 mm LW-PLA shell + spar + glue (assumption;
                                                      # 09a writes 1.1 kg/m^2 but uses 1.1e-4 g/mm^2 = 0.11 kg/m^2)
pd.DataFrame({"mass [g]": {**{k: v * 1000 for k, v in COMMON.items()}, **{f"propulsion: {NAME[k]}": v * 1000 for k, v in UNIT_MASS.items()}}})

## 2. The three MERLINs (Dedalus)

`Merlin` is `FixedWing` with a new fuselage and a `propulsion` switch: `edf` puts notebook 25's duct (bell-mouth,
shroud, area-ruled nozzle, seven stator vanes) on the nose and lets the fuselage run through the nozzle, so the jet
leaves as a ring around it; `tractor` puts the motor and spinner on the nose; `pusher` gives it a clean sensor nose and
puts the motor at the end of the tail cone, the propeller behind the tail. The design file is copied into the run folder,
as in 09a, so a copilot proposal edits the copy.

In [ ]:
design_file = ROOT / "merlin.py"
shutil.copy(Path("designs/merlin.py"), design_file)
merlin = dedalus.load_design(f"{design_file}:Merlin")
display(pd.DataFrame(merlin.params.table()).set_index("name"))
GEOM = {k: merlin.generate(propulsion=k) for k in KINDS}
for k, g in GEOM.items():
    print(f"{NAME[k]:<18s} valid {g.shape.isValid()}, {len(g.shape.Solids())} solid, bounding box {np.round(g.dimensions, 0)} mm")

In [ ]:
try:
    for k in KINDS:
        dviz.show(dviz.plot3d(GEOM[k]))
except ImportError as e:
    print("3D view needs pyvista:", e)
fig, ax = plt.subplots(3, 1, figsize=(13, 7.5), sharex=True)
for a, k in zip(ax, KINDS):
    for poly in mdesign.outline({"propulsion": k})["side"]:
        a.fill(poly[:, 0], poly[:, 1], color=COLOR[k], alpha=0.35, ec="k", lw=0.6)
    a.axvline(mdesign.Merlin.layout(merlin.resolve(propulsion=k))["prop_x"] / 1000, color="k", ls=":", lw=1)
    a.set_aspect("equal"); a.set_title(f"{NAME[k]}: side view (propulsor plane dotted)", fontsize=10); a.grid(alpha=0.3)
ax[-1].set_xlabel("x [m]")
fig.tight_layout()

In [ ]:
fig = dviz.plot_sections(GEOM["edf"], normal="x", positions=[-215.0, -190.0, -150.0, 100.0], cols=4)   # lip, stators, nozzle exit, wing

## 3. Size, mass and drag build-up

The airframe mass comes from the CAD's surface area at 09a's shell density. The parasite drag is a component
build-up (`merlin.drag_buildup`: flat-plate friction on each component's own length times its form factor, plus 10 %
interference) with an allowance for the skid, the sensor intake and the antennas. The EDF nacelle's outside friction
and its jet scrubbing the fuselage are charged to the fan (part 2), the propellers' slipstream to the propellers (the
thrust deduction).

In [ ]:
V_DESIGN = 25.0
p0 = merlin.resolve()
area = GEOM["pusher"].surface_area
COMMON["airframe (printed shells)"] = round(area * SHELL_AREAL_DENSITY_G_MM2 / 1000, 3)
rows = {}
for k in KINDS:
    b = mdesign.drag_buildup({"propulsion": k}, V_DESIGN, NU)
    mass = sum(COMMON.values()) + UNIT_MASS[k]
    S = b["planform"]
    rows[NAME[k]] = {"all-up mass [kg]": mass, "wing area [m²]": S, "aspect ratio": b["aspect_ratio"],
                     "wing loading [N/m²]": mass * G / S, "stall speed (Cl_max 1.2) [m/s]": math.sqrt(2 * mass * G / (RHO * S * 1.2)),
                     "Cd0 build-up": b["cd0"], "Cd0 with allowance": (b["cd_area_m2"] + mf.EXTRA_CD_AREA_M2) / S,
                     **{f"Cd·A {c} [cm²]": v * 1e4 for c, v in b["parts_m2"].items()}}
SIZE = pd.DataFrame(rows)
print(f"airframe from the CAD's surface area: {COMMON['airframe (printed shells)'] * 1000:.0f} g")
SIZE.round(4)

## 4. Wing structure (Talos)

The same model as 09a (solid-equivalent LW-PLA, the centre section clamped, the lift as a pressure on the lower
skins), with MERLIN's load cases. The dash is the new one: at 50–65 m/s a gust hits much harder than at a mapping
drone's cruise. Pratt's gust load factor `Δn = K_g ρ V U a S / (2 W)` (a 7.5 m/s gust, the alleviation
`K_g = 0.88 μ / (5.3 + μ)` with the mass ratio `μ = 2 (W/S) / (ρ c a g)`) at 60 m/s sets it — a light, fast aircraft
feels gusts hard.

| case | loads | represents |
|---|---|---|
| `pull_up` | 3 × W on the lower skins | the turns between the sampling passes and the recovery from a gust upset |
| `dash_gust` | (1 + Δn) × W | a vertical gust at the dash speed |

In [ ]:
LW_PLA = talos.Material("LW-PLA printed wing (solid-equivalent)", youngs_modulus=400.0, poissons_ratio=0.35,
                        density=0.6e-9, yield_strength=12.0, source="assumed (09a); replace with coupon tests")
W_MAX = max(SIZE.loc["all-up mass [kg]"]) * G
S_W = SIZE.loc["wing area [m²]"].iloc[0]
V_GUST, U_GUST, A_LIFT = 60.0, 7.5, 2 * math.pi * 0.85 * 6.5 / (6.5 + 2)
MU = 2 * W_MAX / S_W / (RHO * p0["root_chord"] * (1 + p0["taper"]) / 2000 * A_LIFT * G)
K_G = 0.88 * MU / (5.3 + MU)
N_GUST = 1 + K_G * RHO * V_GUST * U_GUST * A_LIFT * S_W / (2 * W_MAX)
print(f"gust load factor at {V_GUST:.0f} m/s: {N_GUST:.1f} (mass ratio {MU:.1f}, alleviation {K_G:.2f})")

def lower_skins(step):
    info = talos.inspect_step(step, units="mm-N-MPa")
    skins = [s for s in info.surfaces if s.kind == "BSpline surface" and s.area > 5e4]
    return [min((s for s in skins if (s.centroid[1] > 0) == right), key=lambda s: s.centroid[2]).tag for right in (False, True)]

def wing_model(rev, n, name):
    yc = rev.params["fuselage_diameter"] / 2 + 5.0
    regions = [talos.SurfacesInBox("root", (-1.0, -yc - 0.1, -100.0, 400.0, yc + 0.1, 100.0)), talos.Surfaces("lift", lower_skins(rev.step))]
    return talos.StructuralModel(rev.step, "mm-N-MPa", LW_PLA, regions, [talos.FixedSupport("root")],
                                 [talos.Pressure("lift", n * W_MAX / (S_W * 1e6))], talos.MeshSettings(element_size=10.0), name=name)

def pull_up(rev):
    return wing_model(rev, 3.0, "pull_up")

def dash_gust(rev):
    return wing_model(rev, N_GUST, "dash_gust")

ws = core.Workspace.create(ROOT / "workspace", name="MERLIN")
design = ws.add_design("merlin", f"{design_file}:Merlin")
w1 = design.new_revision(part="wing", note="baseline wing NACA 2412")
w1.generate()
w2 = w1.branch(thickness=0.15, note="NACA 2415: thicker wing")
w2.generate()
for rev in (w1, w2):
    for case in (pull_up, dash_gust):
        rev.run_fea(case.__name__, case, progress=True)

def row(rev):
    d = {"note": rev.record.get("note", ""), "thickness": rev.params["thickness"]}
    for case in ("pull_up", "dash_gust"):
        ev = rev.evaluation("fea", case)
        d[f"{case} tip [mm]"] = ev.metrics.get("max_displacement", np.nan) if ev and ev.ok else np.nan
        d[f"{case} SF"] = ev.metrics.get("safety_factor_yield", np.nan) if ev and ev.ok else np.nan
    return d

WING = pd.DataFrame({r.id: row(r) for r in (w1, w2)}).T
display(WING)
if WING["dash_gust SF"].notna().any():
    preferred = ws.revision(WING["dash_gust SF"].astype(float).idxmax())
    preferred.label("preferred", note="higher gust safety factor")
else:
    preferred = w1
    print("FEA not run (CalculiX missing?): the baseline wing stays; see ws.status()")
ws.status()

In [ ]:
ev = preferred.evaluation("fea", "dash_gust")
if ev is not None and ev.ok:
    tviz.show(tviz.plot_results(ev.tool_results[-1], field="von_mises"))
    fig = talos.plot_along_axis(ev.tool_results[-1], axis="y", quantity="displacement", component=2)

## 5. Whole-aircraft aerodynamics (Aeromant, optional)

One coarse RANS run per variant at 2° (the dash flies at a low lift coefficient) — 09a's case with
`reference_length = span/4`. A run replaces that variant's build-up Cd0 by the CFD point through a parabolic polar
(Oswald 0.8). Without OpenFOAM the build-up stays — section 3's numbers.

In [ ]:
V_CFD, AOA = 30.0, 2.0
CFD_POLAR = {}

def cruise(rev, workdir):
    return aeromant.CFDCase("rans_ksst_external", rev.stl,
        dict(velocity=V_CFD, kinematic_viscosity=NU, density=RHO, reference_area=S_W, reference_length=p0["span"] / 4000,
             center_of_rotation=(0.07, 0.0, 0.0), iterations=400, residual_target=1e-4, surface_level=4, near_level=3,
             wake_level=2, cells_per_length=2.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())

if RUN_CFD:
    for k in KINDS:
        rev = preferred.branch(part="aircraft", propulsion=k, angle_of_attack_deg=AOA, note=f"{NAME[k]}, {AOA} deg, CFD")
        rev.generate(stl_tolerance=0.2)
        ev = rev.run_cfd("cruise_2deg", cruise, progress=True)
        if ev.ok:
            cl, cd = ev.metrics["Cl"], ev.metrics["Cd"]
            CFD_POLAR[k] = cd - cl ** 2 / (math.pi * SIZE.loc["aspect ratio", NAME[k]] * 0.8)
            print(f"{NAME[k]}: Cl {cl:.3f}, Cd {cd:.4f} -> Cd0 {CFD_POLAR[k]:.4f} (build-up {SIZE.loc['Cd0 with allowance', NAME[k]]:.4f})")
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): the drag build-up is the polar")

## 6. Print the nose unit (Mellonia)

The EDF's duct with its stator vanes and the ring they stand on, printed upright on the nozzle exit's flat face.

In [ ]:
nose = design.new_revision(part="nose", propulsion="edf", note="EDF duct with stators, for printing")
nose.generate()
ev_p = nose.run_print("exit_down", GENERIC_PLA_0_2MM, mellonia.Orientation(rotate_y=90))
print(ev_p)

# Part 2 — the race to the fire

## 7. The three propulsors

`mf.build_merlins` assembles the three aircraft from the existing models and returns each one's airframe (mass and
polar) and propulsor (net thrust and shaft power over airspeed × rpm):

- **ducted fan** — `boreas.ducted` on notebook 25's 90 mm, 12-blade rotor in MERLIN's duct (its exit area as built:
  `ducted_fan.housing_geometry`), `duct_loss` fitted to the same 3.0 kgf / 1930 W catalogue point as notebook 25, less the
  nacelle's outside friction and the annular jet scrubbing the fairing behind the nozzle;
- **tractor** and **pusher** — Boreas BEMT on notebook 25's 10-inch two-blade planform, the pitch picked from 6, 8 and 10
  inches for the shortest time to a 20 km fire, the rpm limited to a tip Mach number of 0.75, working in the
  installation's effective wake (`boreas.wake.effective_inflow` over `air_propeller.installation_wake`: the fuselage as the
  pod; the wing as the "pylon" ahead of a tractor's slipstream, the fin and both tail-plane halves ahead of a pusher's
  disc) and paying the thrust deduction (`air_propeller.installation_drag`).

In [ ]:
MERLINS = mf.build_merlins(max_electrical_w=MAX_ELECTRICAL_W, drive_efficiency=DRIVE_EFF, common_mass_kg=COMMON,
                           unit_mass_kg=UNIT_MASS, mission=MISSION, distance_m=20000.0, quick=QUICK)
for k, cd0 in CFD_POLAR.items():                              # the CFD's polar where it ran
    MERLINS[k]["airframe"].cd0 = cd0 + mf.EXTRA_CD_AREA_M2 / MERLINS[k]["airframe"].wing_area_m2
info = MERLINS["edf"]["info"]
print(f"EDF: duct_loss fitted {info['duct_loss']:.3f}, exit area ratio {info['housing']['exit_area_ratio']:.3f}, "
      f"jet scrubbing {info['scrub_area_m2'] * 1e4:.0f} cm² of fuselage")
for k in ("tractor", "pusher"):
    i = MERLINS[k]["info"]
    print(f"{NAME[k]}: rpm limit {i['rpm_max']:.0f}; chosen {i['prop'].name}, effective wake w = {i['w']:.4f}, thrust deduction t = {i['t']:.4f}")
    display(pd.DataFrame(i["pitch_study"]).set_index("pitch [in]").round(4))

## 8. Thrust, power and speed

Full throttle against the aircraft's drag (left): where they cross is the top speed, the gap the acceleration and the
climb. The power in level flight (middle) and the energy per kilometre (right) decide how fast MERLIN can afford to fly
when it must come back. A propeller's thrust falls with airspeed; the ducted fan's small fast jet holds its thrust
longer but pays for it in propulsive efficiency `2V / (V + V_jet)` at every speed.

In [ ]:
PERF = {k: mf.performance(v["airframe"], v["unit"]) for k, v in MERLINS.items()}
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for k, pf in PERF.items():
    ax[0].plot(pf["V"], pf["T_full"], color=COLOR[k], lw=2, label=f"{NAME[k]}: full throttle")
    ax[0].plot(pf["V"], pf["D"], color=COLOR[k], lw=1, ls="--")
    ax[1].plot(pf["V"], pf["P_level"], color=COLOR[k], lw=2, label=NAME[k])
    ax[2].plot(pf["V"], pf["P_level"] / pf["V"] / 3.6, color=COLOR[k], lw=2, label=NAME[k])
ax[0].set(xlabel="airspeed [m/s]", ylabel="N", title="net thrust available (solid) and drag (dashed)", ylim=(0, None)); ax[0].legend(fontsize=8)
ax[1].set(xlabel="airspeed [m/s]", ylabel="W", title="electrical power in level flight", ylim=(0, MAX_ELECTRICAL_W * 1.05)); ax[1].legend(fontsize=8)
ax[2].set(xlabel="airspeed [m/s]", ylabel="Wh/km", title="energy per kilometre", ylim=(0, 4)); ax[2].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()
pd.DataFrame({NAME[k]: {"mass [kg]": MERLINS[k]["airframe"].mass_kg, "static thrust [N]": MERLINS[k]["unit"].full(0.0)[0],
                        "top speed [m/s]": pf["v_top"], "best climb [m/s]": pf["roc_max"], "at [m/s]": pf["v_climb"],
                        "best-range speed [m/s]": pf["v_range"], "energy there [Wh/km]": pf["wh_per_km_best"],
                        "stall [m/s]": pf["v_stall"]} for k, pf in PERF.items()}).round(2)

## 9. Time to the fire — 5, 8, 10, 20 and 30 km

`mf.race`: off the stand at 12 m/s, a full-power climb to 150 m at the best-climb speed, a full-power acceleration,
then the dash — at the top speed if the battery allows it, otherwise at the fastest speed that still leaves the energy
for three minutes of sampling, the flight home at the best-range speed and a 15 % reserve. Still air. The race is
won in one of two ways: close in by **thrust** (the top speed), far out by **energy** (the efficiency at speed).

In [ ]:
D_ALL = np.linspace(5.0, 40.0, 36)
RACE = {k: [mf.race(v["airframe"], v["unit"], d * 1000, MISSION) for d in D_ALL] for k, v in MERLINS.items()}
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for k, rr in RACE.items():
    t = [r["time_to_fire_s"] / 60 if r["reachable"] else np.nan for r in rr]
    v = [r["dash_speed"] if r["reachable"] else np.nan for r in rr]
    ax[0].plot(D_ALL, t, color=COLOR[k], lw=2, label=NAME[k]); ax[1].plot(D_ALL, v, color=COLOR[k], lw=2, label=NAME[k])
for d in DISTANCES_KM:
    ax[0].axvline(d, color="#bbbbbb", lw=0.8, zorder=0); ax[1].axvline(d, color="#bbbbbb", lw=0.8, zorder=0)
ax[0].set(xlabel="distance to the fire [km]", ylabel="minutes", title="time from launch to overhead the fire"); ax[0].legend()
ax[1].set(xlabel="distance to the fire [km]", ylabel="m/s", title="dash airspeed it can afford"); ax[1].legend()
RACE_TAB = {}
for k, v in MERLINS.items():
    for d in DISTANCES_KM:
        r = mf.race(v["airframe"], v["unit"], d * 1000, MISSION)
        RACE_TAB[(f"{d:.0f} km", NAME[k])] = ({"time to the fire [min]": r["time_to_fire_s"] / 60, "dash speed [m/s]": r["dash_speed"],
                                               "limited by": r["limit"], "energy out [Wh]": r["energy_out_wh"],
                                               "sampling [Wh]": r["energy_sampling_wh"], "home [Wh]": r["energy_home_wh"],
                                               "home in [min]": r["home_time_s"] / 60} if r["reachable"]
                                              else {"time to the fire [min]": np.nan, "limited by": r["why"]})
RACE_TAB = pd.DataFrame(RACE_TAB).T
x = np.arange(len(DISTANCES_KM))
for i, k in enumerate(KINDS):
    vals = [RACE_TAB.loc[(f"{d:.0f} km", NAME[k]), "time to the fire [min]"] for d in DISTANCES_KM]
    ax[2].bar(x + (i - 1) * 0.27, vals, 0.25, color=COLOR[k], label=NAME[k])
    for xx, vv in zip(x + (i - 1) * 0.27, vals):
        ax[2].text(xx, vv, f"{vv:.1f}", ha="center", va="bottom", fontsize=7)
ax[2].set_xticks(x, [f"{d:.0f} km" for d in DISTANCES_KM]); ax[2].set(ylabel="minutes", title="time to the fire"); ax[2].legend()
for a in ax:
    a.grid(alpha=0.3)
fig.tight_layout()
RACE_TAB.round(2)

In [ ]:
WINNER = {}
for d in DISTANCES_KM:
    sub = RACE_TAB.xs(f"{d:.0f} km", level=0)["time to the fire [min]"].astype(float)
    best = sub.idxmin()
    WINNER[d] = next(k for k in KINDS if NAME[k] == best)
    others = ", ".join(f"{n} +{(t - sub.min()) * 60:.0f} s" for n, t in sub.items() if n != best)
    print(f"{d:4.0f} km: fastest is the {best} ({sub.min():.1f} min); {others}")
FASTEST = WINNER[20.0]          # the design point: the 20 km case (override here)
print(f"\nMERLIN's propulsion: {NAME[FASTEST]}")

**Reading the race.** The two propellers are within a few percent of each other: the pusher works in the slower
air behind the fuselage and the tail (its effective wake is a gain) and pays a smaller thrust deduction than the tractor,
whose slipstream scrubs the whole fuselage and the wing root. The ducted fan is close only where it can dash flat out
the whole way; past that its energy per kilometre — a small, fast jet — forces it to slow down to keep the way home, and
it falls far behind. What the numbers leave out: the pusher's propeller cuts the tail's wakes (blade vibration and a
louder tone, notebook 25 §10–11), and needs the belly landing to keep the blades off the ground (a folding propeller);
the EDF's protected rotor and its tolerance of smoke and debris.

## 10. Where the sensor breathes

The gas sensor must sample air the propulsor has not touched. On the **pusher** the nose is clean. On the **tractor**
the nose is the propeller: the sensor sits in a wing-root intake, inside the slipstream, which mixes the plume's
structure. The **EDF**'s inlet swallows a stream tube several times its own area at low speed — a sensor port in the lip
reads the air being drawn in, not the free stream ahead.

In [ ]:
from vegeta.boreas import ducted
fan = MERLINS["edf"]["info"]["fan"]
rows = {}
for V in (15.0, 30.0, 50.0):
    e = ducted.solve(fan, boreas.Airfoil(**mf.SECTION_KW), MERLINS["edf"]["unit"].full(V)[2], V, RHO)
    rows[f"{V:.0f} m/s"] = {"EDF capture area / highlight area": e.mass_flow / (RHO * V) / (math.pi * (MERLINS['edf']['info']['housing']['highlight_radius'] / 1000) ** 2),
                            "EDF jet speed [m/s]": e.exit_velocity}
    pr = MERLINS["tractor"]["info"]["prop"]
    o = boreas.solve(pr, boreas.Airfoil(**mf.SECTION_KW), MERLINS["tractor"]["unit"].full(V)[2], V, RHO)
    rows[f"{V:.0f} m/s"]["tractor slipstream excess, far wake [m/s]"] = 2 * float(np.average(o.induced_velocity, weights=o.r))
pd.DataFrame(rows).round(2)

# Part 3 — the winner in flight

## 11. The whole aircraft with its propeller (Aeromant, optional)

The winner's airframe at 2° with its propeller as one rotor disk (`hull_rotor_disk`: the single-disk case of
`aircraft_rotor_disks`, blade-element source terms from the Boreas blade and polar) at the 20 km race's dash speed,
against the airframe alone. Only for a propeller: the EDF's rotor sits inside its duct, a case for `rotor_mrf_installed`
(notebook 25 §13.5).

In [ ]:
cfd_disk = None
if RUN_CFD and FASTEST != "edf":
    prop = MERLINS[FASTEST]["info"]["prop"]
    r20 = mf.race(MERLINS[FASTEST]["airframe"], MERLINS[FASTEST]["unit"], 20000, MISSION)
    rpm = MERLINS[FASTEST]["unit"].full(r20["dash_speed"])[2]
    rev = preferred.branch(part="aircraft", propulsion=FASTEST, angle_of_attack_deg=AOA, note=f"{NAME[FASTEST]} with its rotor disk")
    rev.generate(stl_tolerance=0.2)
    th = math.radians(AOA)
    lay = mdesign.Merlin.layout(rev.params)
    x, z = lay["prop_x"], lay["axis_z"]
    centre = [(x * math.cos(th) + z * math.sin(th)) / 1000, 0.0, (-x * math.sin(th) + z * math.cos(th)) / 1000]
    axis = [-math.cos(th), 0.0, math.sin(th)]
    sec = boreas.Airfoil(**mf.SECTION_KW)
    alphas = np.unique(np.r_[np.arange(-180, -30, 10), np.arange(-30, 31, 1), np.arange(40, 181, 10)]).astype(float)
    cl_, cd_ = sec.coefficients(np.radians(alphas))
    disk = aeromant.CFDCase("hull_rotor_disk", rev.stl,
        dict(velocity=r20["dash_speed"], kinematic_viscosity=NU, density=RHO, reference_area=S_W, reference_length=p0["span"] / 6000,
             center_of_rotation=(0.07, 0.0, 0.0), iterations=600, residual_target=1e-4, surface_level=4, near_level=3, wake_level=2,
             cells_per_length=2.0, disk1_center=centre, disk_axis=axis, diameter=prop.diameter, rpm=rpm, blades=prop.blades,
             blade=[[r_, b_, c_] for r_, b_, c_ in zip(prop.r, prop.beta_deg, prop.chord)],
             polar=[[a_, float(d_), float(l_)] for a_, l_, d_ in zip(alphas, cl_, cd_)], rotation1=1, disk_level=5),
        workdir=ROOT / "cfd_disk", geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    print(disk.prepare(overwrite=True))
    cfd_disk = disk.run(progress=True)
    print(cfd_disk)
    if cfd_disk.ok:
        aviz.show(aviz.plot_field_slice(disk, "U", normal="y", origin=(0, 0.0, 0)))
        clip = aviz.animate_particles(disk, OUT / f"merlin_{FASTEST}_flow.mp4", n=500, seconds=6, fps=24)
        display(Video(str(clip), embed=False, width=900))
else:
    print("rotor-disk CFD skipped" + (" (VEGETA_SKIP_OPENFOAM=1)" if not RUN_CFD else " (the winner is the ducted fan)"))

## 12. The mission through the smoke

A 40 MW fire (a front of a few tens of metres in a forest) 20 km east of the launch site, a 5 m/s south-west wind
carrying the column north-east. The smoke is a Gaussian plume (`mf.Plume`: Briggs buoyant rise, Pasquill–Gifford
class C dispersion, CO at 0.10 kg per kg of fuel). MERLIN dashes at the race's speed, flies crosswind passes 600 m
downwind at four heights around 150 m, and goes home at the best-range speed into the wind. The sensor has a one-second
time constant.

In [ ]:
PLUME = mf.Plume(source_xy=(20000.0, 0.0), heat_mw=40.0, wind_speed=5.0, wind_from_deg=225.0)
FOREST = mf.Forest()
af, unit = MERLINS[FASTEST]["airframe"], MERLINS[FASTEST]["unit"]
R20 = mf.race(af, unit, 20000.0, MISSION)
EP = mf.fly(af, unit, FOREST, PLUME, race_result=R20, mission=MISSION)
print(json.dumps({k: v for k, v in EP.summary().items()}, default=float, indent=1))
display(EP.phase_table().round(1))
fig = mf.profile_figure(EP, PLUME)

### Is it real, and how big?

Each pass integrates the CO across the column at its height; a Gaussian with ground reflection through the four
integrals gives the CO flux, the plume's height and its depth — and with the emission factor and the fuel's heat of
combustion, the fire's heat release. The model's own fire is 40 MW: the retrieval is checked against the truth here; in
the field the wind speed is measured on board.

In [ ]:
EST = mf.source_estimate(PLUME, EP.passes)
zs = np.linspace(0, 400, 200)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for p_ in EP.passes:
    s_, c_ = PLUME.local(p_["x"], p_["y"])
    ax[0].plot(c_, p_["ppm"], lw=1.5, label=f"{p_['name']} at {np.mean(p_['z_agl']):.0f} m")
ax[0].set(xlabel="crosswind position [m]", ylabel="CO above background [ppm]", title="the passes"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot(EST["cwic_kg_m2"] * 1e6, EST["pass_heights_m"], "o", color="k", label="passes")
ax[1].plot(PLUME.cwic(EST["downwind_m"], zs, EST["co_kg_s"], EST["plume_height_m"], EST["sigma_z_m"]) * 1e6, zs, color=COLOR[FASTEST], label="fit")
ax[1].plot(PLUME.cwic(EST["downwind_m"], zs) * 1e6, zs, ":", color="#555555", label="the model's truth")
ax[1].set(xlabel="crosswind-integrated CO [mg/m²]", ylabel="height AGL [m]", title="through the column"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()
pd.Series({"peak CO [ppm]": EST["peak_ppm"], "CO flux [g/s]": EST["co_kg_s"] * 1000, "true CO flux [g/s]": PLUME.co_kg_s * 1000,
           "plume height [m]": EST["plume_height_m"], "fire heat release [MW]": EST["heat_mw"], "true [MW]": PLUME.heat_mw,
           "confirmed": EST["peak_ppm"] > 10 * 0.05}).to_frame("ignition check")

## 13. The movie — with the smoke

The forest around the aircraft, the smoke column as particles drawn from the plume model (darker and denser near the
fire, paler and wider downwind), MERLIN and its trail, the map and the sensor's reading. Time-lapse: half the movie is
the passes through the smoke.

In [ ]:
MOVIE, FRAMES = mf.render_movie(EP, FOREST, PLUME, OUT / f"merlin_{FASTEST}_smoke_20km.mp4", seconds=1.0 if QUICK else 24.0,
                                fps=10 if QUICK else 20, title=f"MERLIN ({NAME[FASTEST]}) — a fire 20 km out", progress=True)
fig, axes = plt.subplots(2, 3, figsize=(18, 6.2))
for a, fr in zip(axes.flat, np.linspace(0, len(FRAMES) - 1, 6).astype(int)):
    a.imshow(FRAMES[fr][:, :, ::-1]); a.axis("off")
fig.tight_layout()
Video(str(MOVIE), embed=False, width=960)

## 14. Hand-off

In [ ]:
handoff = {
    "source": "26_merlin", "propulsion": FASTEST, "distances_km": list(DISTANCES_KM),
    "winner_per_distance": {f"{d:.0f} km": WINNER[d] for d in DISTANCES_KM},
    "race": {f"{i[0]} / {i[1]}": {k: (v if not isinstance(v, (np.floating,)) else float(v)) for k, v in r.items()} for i, r in RACE_TAB.iterrows()},
    "masses_kg": {NAME[k]: MERLINS[k]["airframe"].mass_kg for k in KINDS},
    "polar": {NAME[k]: {"cd0": MERLINS[k]["airframe"].cd0, "S": MERLINS[k]["airframe"].wing_area_m2, "AR": MERLINS[k]["airframe"].aspect_ratio} for k in KINDS},
    "mission": EP.summary(), "source_estimate": {k: v for k, v in EST.items() if not isinstance(v, np.ndarray)},
    "preferred_wing": {"revision": preferred.id, "thickness": preferred.params["thickness"]},
    "movie": str(MOVIE),
}
Path("_runs/merlin_handoff.json").write_text(json.dumps(handoff, indent=1, default=str))
print("written _runs/merlin_handoff.json")

**Next steps:** a folding pusher propeller and its blade under the tail's wakes (notebook 25's pusher sections on this
propeller); the race in a head- and tailwind; the sensor's inlet in CFD; a wind estimate on board for the flux; the
battery's voltage sag at 12 C; the landing skid and the belly-landing load case.